# 1. Project Introduction

# EuroSAT Land-Use and Land-Cover Classification

## Project Title

Reproducing a Deep Learning Baseline and Evaluating a Smaller Network

## Reference Paper

Helber, P., Bischke, B., Dengel, A., and Borth, D. (2019).  
*EuroSAT: A Novel Dataset and Deep Learning Benchmark for Land Use and Land Cover Classification.*  
IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing, 12(7), 2217–2226.

[https://doi.org/10.1109/JSTARS.2019.2918242](https://doi.org/10.1109/JSTARS.2019.2918242)

## Project Overview

This project investigates deep-learning-based land-use and land-cover classification using the EuroSAT RGB satellite image dataset.

The first objective is to reproduce the ResNet-50 baseline reported in the reference paper as closely as possible. After establishing the baseline, we will evaluate a smaller ResNet-18 network to investigate whether it can achieve comparable classification performance with lower computational cost.

## Objectives

1. Explore and verify the EuroSAT RGB dataset.
2. Reproduce the ResNet-50 baseline described in the reference paper.
3. Prepare reproducible training, validation, and test sets.
4. Train and evaluate ResNet-50 using the paper's reported training strategy.
5. Train and evaluate ResNet-18 under comparable experimental conditions.
6. Compare classification performance and computational cost.
7. Analyse classification errors and discuss the experimental results.

## Dataset Source

The primary dataset is the EuroSAT RGB archive published on Zenodo:

https://zenodo.org/records/7711810

The RGB dataset contains 27,000 labelled images across 10 land-use and land-cover classes. Each image has a spatial resolution of 64 × 64 pixels with three RGB channels.


# 2. Environment Setup

This section records the software environment used for the experiments and checks whether GPU acceleration is available.

Recording the software versions and hardware helps make the experiments easier to reproduce.


In [ ]:
import os
import random
import platform
import numpy as np
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("NumPy version:", np.__version__)

# Check GPU availability
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    print("Running on CPU.")

print("Selected device:", DEVICE)


# 3. Dataset Acquisition and Verification

## 3.1 Dataset Source

The EuroSAT RGB dataset is obtained from the official Zenodo record:

https://zenodo.org/records/7711810

The dataset contains 27,000 labelled satellite image patches belonging to 10 land-use and land-cover classes. Each image has a spatial resolution of 64 × 64 pixels with three RGB colour channels.


### 3.2 Dataset Download and Verification

We downloaded the dataset from Zenodo and verified its integrity using an MD5 checksum. The checksum matched the expected value, confirming that the downloaded archive passed the verification check.

In [ ]:
import os
import hashlib
import shutil
import urllib.request
import zipfile
from pathlib import Path

# Official Zenodo EuroSAT RGB dataset
DATASET_URL = (
    "https://zenodo.org/api/records/7711810/"
    "files/EuroSAT_RGB.zip/content"
)

ZIP_PATH = Path("/content/EuroSAT_RGB.zip")
DATASET_DIR = Path("/content/EuroSAT_RGB")
TEMP_PATH = Path("/content/EuroSAT_RGB.zip.part")
EXTRACT_DIR = Path("/content/EuroSAT_extract")

EXPECTED_MD5 = "f46e308c4d50d4bf32fedad2d3d62f3b"
EXPECTED_TOTAL_IMAGES = 27000

EXPECTED_CLASSES = [
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake",
]

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}


def calculate_md5(file_path):
    """Calculate the MD5 checksum of a file."""
    md5 = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            md5.update(chunk)

    return md5.hexdigest()


def count_images(folder):
    """Count images directly inside a class folder."""
    if not folder.is_dir():
        return 0

    return sum(
        1 for path in folder.iterdir()
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def dataset_is_valid(root):
    """Check class folders and total image count."""
    if not root.is_dir():
        return False

    counts = {
        name: count_images(root / name)
        for name in EXPECTED_CLASSES
    }

    return (
        all(count > 0 for count in counts.values())
        and sum(counts.values()) == EXPECTED_TOTAL_IMAGES
    )


# Step 1: Verify or download the official Zenodo archive.
archive_valid = (
    ZIP_PATH.is_file()
    and calculate_md5(ZIP_PATH) == EXPECTED_MD5
)

if not archive_valid:
    print("Downloading official EuroSAT RGB from Zenodo...")

    try:
        request = urllib.request.Request(
            DATASET_URL,
            headers={"User-Agent": "EuroSAT-Project/1.0"}
        )

        with urllib.request.urlopen(
            request, timeout=180
        ) as response:
            print("HTTP status:", response.status)

            with open(TEMP_PATH, "wb") as file:
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    file.write(chunk)

        actual_md5 = calculate_md5(TEMP_PATH)

        if actual_md5 != EXPECTED_MD5:
            raise ValueError(
                "Zenodo MD5 mismatch. "
                f"Expected {EXPECTED_MD5}, got {actual_md5}."
            )

        os.replace(TEMP_PATH, ZIP_PATH)
        print("Official Zenodo archive downloaded and verified.")

    finally:
        if TEMP_PATH.exists():
            TEMP_PATH.unlink()

else:
    print("Verified Zenodo archive already exists.")

# Verify archive integrity before extraction.
if calculate_md5(ZIP_PATH) != EXPECTED_MD5:
    raise ValueError("The Zenodo archive failed MD5 verification.")

if not zipfile.is_zipfile(ZIP_PATH):
    raise ValueError("The downloaded archive is not a valid ZIP file.")

print("Archive size:", round(ZIP_PATH.stat().st_size / 1_000_000, 2), "MB")
print("Verified MD5:", calculate_md5(ZIP_PATH))

# Step 2: Reuse a valid extracted dataset only after verifying the ZIP.
if dataset_is_valid(DATASET_DIR):
    print("Dataset folder is valid. Skipping extraction.")

else:
    print("Dataset folder missing or incomplete. Extracting Zenodo archive...")

    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)

    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        # Prevent unexpected paths from escaping the extraction folder.
        base = EXTRACT_DIR.resolve()

        for member in archive.infolist():
            target = (EXTRACT_DIR / member.filename).resolve()

            if target != base and base not in target.parents:
                raise ValueError(
                    f"Unsafe archive path detected: {member.filename}"
                )

        archive.extractall(EXTRACT_DIR)

    # Locate the directory containing all expected class folders.
    candidates = [EXTRACT_DIR] + [
        path for path in EXTRACT_DIR.rglob("*")
        if path.is_dir()
    ]

    source_dir = next(
        (
            folder for folder in candidates
            if all(
                (folder / name).is_dir()
                for name in EXPECTED_CLASSES
            )
        ),
        None
    )

    if source_dir is None:
        raise RuntimeError(
            "Could not find all ten class folders after extraction."
        )

    # Replace any previous reconstructed dataset with the verified archive.
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)

    if source_dir == EXTRACT_DIR:
        # The class folders are directly inside the extraction directory.
        shutil.copytree(source_dir, DATASET_DIR)
    else:
        shutil.copytree(source_dir, DATASET_DIR)

    shutil.rmtree(EXTRACT_DIR)

# Step 3: Final dataset verification.
if not dataset_is_valid(DATASET_DIR):
    raise RuntimeError(
        "Extracted dataset verification failed. "
        "Expected 27,000 images across the ten classes."
    )

print("\nFINAL DATASET VERIFICATION")
print("-" * 40)

total_images = 0

for class_name in EXPECTED_CLASSES:
    count = count_images(DATASET_DIR / class_name)
    total_images += count
    print(f"{class_name}: {count:,}")

print("-" * 40)
print("Total images:", total_images)
print("Number of classes:", len(EXPECTED_CLASSES))
print("Dataset directory:", DATASET_DIR)
print("Source: Official Zenodo EuroSAT RGB archive")

assert total_images == EXPECTED_TOTAL_IMAGES
assert dataset_is_valid(DATASET_DIR)

print("\nAll dataset checks passed!")



# 4. Dataset Exploration

In this section, we explore the EuroSAT RGB dataset to understand its class distribution, image characteristics, and visual differences between land-use and land-cover categories.

We display the image counts for each class and examine representative sample images before preparing the dataset for model training.

The dataset contains 27,000 images across 10 classes, with each image measuring 64 × 64 pixels and containing three RGB channels.

The dataset is released under the MIT licence.

We also consider dataset limitations reported in the reference paper, including colour casts caused by the lack of atmospheric correction and image patches containing clouds or possible mislabelling. These limitations will be revisited during the model's error analysis.



### 4.1 Inspecting Sample Images

We selected three representative images from each of the 10 classes to examine their visual characteristics and differences. We also checked the image dimensions and colour mode to understand how the satellite images are represented in the dataset.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# Display three sample images from each class
class_names = sorted(EXPECTED_CLASSES)

fig, axes = plt.subplots(
    nrows=len(class_names),
    ncols=3,
    figsize=(12, 30)
)

for row, class_name in enumerate(class_names):

    class_path = os.path.join(DATASET_DIR, class_name)

    image_files = sorted([
        file_name
        for file_name in os.listdir(class_path)
        if file_name.lower().endswith((".jpg", ".jpeg", ".png"))
    ])

    # Select three sample images
    sample_images = image_files[:3]

    for col, file_name in enumerate(sample_images):

        image_path = os.path.join(class_path, file_name)

        with Image.open(image_path) as image:
            axes[row, col].imshow(image.convert("RGB"))

        axes[row, col].axis("off")

        if col == 0:
            axes[row, col].set_title(
                class_name,
                loc="left",
                fontsize=11,
                fontweight="bold"
            )

plt.tight_layout()
plt.show()

### 4.2 Class Distribution

We visualised the number of images in each land-use and land-cover class using a bar chart. This provides an overview of the class distribution and helps us understand whether the dataset is balanced.

In [ ]:
from collections import Counter

# Count images in each class
class_counts = {}

for class_name in sorted(EXPECTED_CLASSES):
    class_path = os.path.join(DATASET_DIR, class_name)

    image_files = [
        file_name
        for file_name in os.listdir(class_path)
        if file_name.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    class_counts[class_name] = len(image_files)

# Plot class distribution
plt.figure(figsize=(12, 6))

plt.bar(
    class_counts.keys(),
    class_counts.values()
)

plt.title("Number of Images per EuroSAT Class")
plt.xlabel("Land-Use and Land-Cover Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

### 4.3 Checking Image Data Quality

We checked whether the image files could be opened successfully and whether they had the expected RGB format and image dimensions. This helped us identify any unreadable or incorrectly formatted images before training our models.

In [ ]:
unreadable_images = []
unexpected_images = []

for class_name in sorted(EXPECTED_CLASSES):
    class_path = os.path.join(DATASET_DIR, class_name)

    for file_name in os.listdir(class_path):
        if not file_name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        image_path = os.path.join(class_path, file_name)

        try:
            with Image.open(image_path) as image:
                image.verify()

            with Image.open(image_path) as image:
                if image.size != (64, 64) or image.mode != "RGB":
                    unexpected_images.append(
                        (image_path, image.size, image.mode)
                    )

        except Exception:
            unreadable_images.append(image_path)

print("Unreadable images:", len(unreadable_images))
print("Images with unexpected dimensions or colour mode:",
      len(unexpected_images))

if unreadable_images:
    print("Examples of unreadable images:", unreadable_images[:5])

if unexpected_images:
    print("Examples of unexpected images:", unexpected_images[:5])


### 4.4 Inspecting Image Properties

We examined the dimensions, colour mode, and pixel values of a sample image to better understand how the satellite images are represented before preprocessing.

In [ ]:
sample_class = "AnnualCrop"
sample_folder = os.path.join(DATASET_DIR, sample_class)

sample_files = sorted([
    file_name
    for file_name in os.listdir(sample_folder)
    if file_name.lower().endswith((".jpg", ".jpeg", ".png"))
])

sample_path = os.path.join(sample_folder, sample_files[0])

with Image.open(sample_path) as image:
    sample_image = image.convert("RGB")
    image_array = np.array(sample_image)

print("Sample image:", sample_files[0])
print("Image dimensions:", sample_image.size)
print("Colour mode:", sample_image.mode)
print("Array shape:", image_array.shape)
print("Data type:", image_array.dtype)
print("Minimum pixel value:", image_array.min())
print("Maximum pixel value:", image_array.max())


### 4.5 Checking for Duplicate Images

We checked the dataset for exact duplicate image files using their MD5 hashes. This helped us identify whether identical files were present under different filenames.

In [ ]:
import hashlib
from collections import defaultdict

image_hashes = defaultdict(list)

for class_name in sorted(EXPECTED_CLASSES):
    class_path = os.path.join(DATASET_DIR, class_name)

    for file_name in os.listdir(class_path):
        if not file_name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        image_path = os.path.join(class_path, file_name)

        with open(image_path, "rb") as file:
            file_hash = hashlib.md5(file.read()).hexdigest()

        image_hashes[file_hash].append((class_name, file_name))

duplicate_groups = [
    files for files in image_hashes.values()
    if len(files) > 1
]

duplicate_file_count = sum(
    len(files) - 1 for files in duplicate_groups
)

print("Total image files checked:", sum(class_counts.values()))
print("Duplicate groups found:", len(duplicate_groups))
print("Extra duplicate files:", duplicate_file_count)

if duplicate_groups:
    print("\nExamples of duplicates:")
    for group in duplicate_groups[:10]:
        print(group)
else:
    print("No exact duplicate files found.")


### 4.6 Dataset-Wide Pixel and Missing-Value Checks

We checked the pixel-value range and missing pixel values across all readable images. We also verified that every image has a valid class label from its folder name.

In [ ]:
pixel_min = []
pixel_max = []
missing_pixels = 0
checked_images = 0

# Check whether the dataset contains only the expected class folders
actual_classes = {
    name for name in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, name))
}

invalid_labels = actual_classes - set(EXPECTED_CLASSES)

for class_name in sorted(EXPECTED_CLASSES):
    class_path = os.path.join(DATASET_DIR, class_name)

    for file_name in os.listdir(class_path):
        if not file_name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        image_path = os.path.join(class_path, file_name)

        with Image.open(image_path) as image:
            pixels = np.array(image.convert("RGB"))

        pixel_min.append(pixels.min())
        pixel_max.append(pixels.max())
        missing_pixels += np.isnan(pixels).sum()
        checked_images += 1

print("Images checked:", checked_images)
print("Dataset-wide minimum pixel value:", min(pixel_min))
print("Dataset-wide maximum pixel value:", max(pixel_max))
print("Missing pixel values:", missing_pixels)
print("Unexpected class folders:", invalid_labels)

assert checked_images == 27000
assert len(invalid_labels) == 0
assert missing_pixels == 0

print("\nDataset-wide checks passed!")

# 5. Data Preparation

After exploring the dataset and checking its quality, we prepared the image paths and corresponding class labels for model training.

We divided the dataset into training, validation, and test sets using a stratified split to preserve the class distribution.

Following the reference paper, 80% of the images are reserved for training and validation, while 20% are held out for testing. The validation set is derived from the training portion and is used for model selection and tuning.

The test set is kept separate and is used for final evaluation after model development.

### 5.1 Collecting Image Paths and Labels

We collected the paths of all images along with their corresponding class labels. These labels will be used as the target values during model training.

In [ ]:
image_paths = []
image_labels = []

for class_name in sorted(EXPECTED_CLASSES):
    class_path = os.path.join(DATASET_DIR, class_name)

    for file_name in sorted(os.listdir(class_path)):
        if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(class_path, file_name))
            image_labels.append(class_name)

print("Total image paths:", len(image_paths))
print("Total labels:", len(image_labels))
print("First image path:", image_paths[0])
print("First image label:", image_labels[0])


# 5. Data Preparation

After exploring the dataset and checking its quality, we prepared the image paths and corresponding class labels for model training.

We divided the dataset into training, validation, and test sets using a stratified split to preserve the class distribution.

Following the reference paper, 80% of the images are reserved for training and validation, while 20% are held out for testing. The validation set is derived from the training portion and is used for model selection and tuning.

The test set is kept separate and is used for final evaluation after model development.

### 5.1 Collecting Image Paths and Labels

We collected the paths of all images along with their corresponding class labels. These labels will be used as the target values during model training.

In [ ]:
image_paths = []
image_labels = []

for class_name in sorted(EXPECTED_CLASSES):
    class_path = os.path.join(DATASET_DIR, class_name)

    for file_name in sorted(os.listdir(class_path)):
        if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(class_path, file_name))
            image_labels.append(class_name)

print("Total image paths:", len(image_paths))
print("Total labels:", len(image_labels))
print("First image path:", image_paths[0])
print("First image label:", image_labels[0])

### 5.2 Splitting the Dataset

We divided the 27,000 EuroSAT images into training, validation, and test sets using stratified sampling. This approach preserves the proportion of each land-use and land-cover class across all three sets.

- **Training set (72%):** 19,440 images used to train the models.
- **Validation set (8%):** 2,160 images used to monitor model performance and guide model selection.
- **Test set (20%):** 5,400 images reserved for final evaluation.

The training and validation sets together account for 80% of the dataset, following the train/test split used in the reference paper. The validation set is derived from the 80% training portion.

We use a fixed random seed of 42 to make the split reproducible. Both ResNet-50 and ResNet-18 will use the same split to ensure a fair comparison.

#### Split Integrity and Class Distribution Checks

After splitting the dataset, we verify the sizes of all three sets and check for overlapping image paths. We confirm that no images are shared between the training, validation, and test sets and that all 27,000 images are accounted for.

We also examine the class distribution in each split to confirm that stratified sampling preserves the expected class proportions.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Fixed random seed for reproducibility
RANDOM_SEED = 42

# --------------------------------------------------
# Step 1: Split the dataset into 80% train/validation
#         and 20% test sets
# --------------------------------------------------

X_train_val, X_test, y_train_val, y_test = train_test_split(
    image_paths,
    image_labels,
    test_size=0.20,              # 20% reserved for testing
    random_state=RANDOM_SEED,
    stratify=image_labels        # Preserve class proportions
)

# --------------------------------------------------
# Step 2: Split the remaining 80% into training
#         and validation sets
# --------------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.10,              # 10% of 80% = 8% of full dataset
    random_state=RANDOM_SEED,
    stratify=y_train_val         # Preserve class proportions
)

# --------------------------------------------------
# Step 3: Display and verify split sizes
# --------------------------------------------------

print("DATASET SPLIT SIZES")
print("-" * 35)

print("Training images:", len(X_train))
print("Validation images:", len(X_val))
print("Test images:", len(X_test))
print("Total images:", len(X_train) + len(X_val) + len(X_test))

# Verify the expected split sizes
assert len(X_train) == 19440
assert len(X_val) == 2160
assert len(X_test) == 5400

# --------------------------------------------------
# Step 4: Check for overlap between dataset splits
# --------------------------------------------------

train_paths = set(X_train)
val_paths = set(X_val)
test_paths = set(X_test)

# Combine all paths to verify that every image is accounted for
all_split_paths = train_paths | val_paths | test_paths

print("\nSPLIT INTEGRITY CHECKS")
print("-" * 35)

print("Train/validation overlap:", len(train_paths & val_paths))
print("Train/test overlap:", len(train_paths & test_paths))
print("Validation/test overlap:", len(val_paths & test_paths))
print("Unique image paths:", len(all_split_paths))

# Verify that there are no overlapping images
assert not (train_paths & val_paths)
assert not (train_paths & test_paths)
assert not (val_paths & test_paths)

# Verify that all 27,000 images are accounted for
assert len(all_split_paths) == 27000

# --------------------------------------------------
# Step 5: Display class distribution across splits
# --------------------------------------------------

# Combine the class labels and their corresponding split names
split_labels = pd.DataFrame({
    "class_name": (
        y_train
        + y_val
        + y_test
    ),
    "split": (
        ["train"] * len(y_train)
        + ["validation"] * len(y_val)
        + ["test"] * len(y_test)
    )
})

# Create a table showing the number of images per class in each split
class_distribution = pd.crosstab(
    split_labels["class_name"],
    split_labels["split"]
)

# Arrange the columns in the correct order
class_distribution = class_distribution[
    ["train", "validation", "test"]
]

print("\nCLASS DISTRIBUTION")
print("-" * 35)
print(class_distribution)

# --------------------------------------------------
# Step 6: Confirm that all checks passed
# --------------------------------------------------

print("\nAll dataset splitting and integrity checks passed!")

### 5.3 Saving the Dataset Splits and Class Mapping

We saved the dataset split assignments in a CSV file containing each image's relative path, class name, numerical class index, and split assignment. We also saved the class-to-index mapping in a JSON file.

Saving these files allows us to reuse the same data splits and consistent class labels across both models, supporting reproducibility and a fair comparison.

In [ ]:
import os
import json
import pandas as pd

# Create a consistent mapping of class names to numerical indices
class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(sorted(EXPECTED_CLASSES))
}

# Collect image paths, labels, and split assignments
records = []

split_data = [
    ("train", X_train, y_train),
    ("validation", X_val, y_val),
    ("test", X_test, y_test)
]

for split_name, paths, labels in split_data:
    for path, label in zip(paths, labels):
        # Save a relative path instead of a Colab-specific absolute path
        relative_path = os.path.relpath(path, DATASET_DIR)

        records.append({
            "relative_path": relative_path,
            "class_name": label,
            "class_idx": class_to_idx[label],
            "split": split_name
        })

# Save the split assignments
splits_df = pd.DataFrame(records)
splits_df.to_csv("/content/eurosat_splits.csv", index=False)

# Save the class-to-index mapping
with open("/content/class_to_idx.json", "w") as file:
    json.dump(class_to_idx, file, indent=2)

# Display a summary
print("Split assignments saved to: /content/eurosat_splits.csv")
print("Class mapping saved to: /content/class_to_idx.json")

print("\nNumber of records:", len(splits_df))
print("\nImages per split:")
print(splits_df["split"].value_counts())

print("\nClass-to-index mapping:")
print(class_to_idx)

print("\nFirst five records:")
print(splits_df.head())


### 5.4 Preparing Images for Model Training

Before training the deep learning models, we prepared the data-loading pipeline in PyTorch. We reused the saved dataset split assignments and class-to-index mapping to maintain consistent image assignments and labels throughout the experiments.

In [ ]:
import os
import json
import pandas as pd

# Load the saved dataset split assignments
splits_df = pd.read_csv("/content/eurosat_splits.csv")

# Load the saved class-to-index mapping
with open("/content/class_to_idx.json", "r") as file:
    class_to_idx = json.load(file)

# Define the dataset directory
DATASET_DIR = "/content/EuroSAT_RGB"

# --------------------------------------------------
# Step 1: Display the saved split information
# --------------------------------------------------

print("Total saved records:", len(splits_df))

print("\nImages per split:")
print(splits_df["split"].value_counts())

print("\nNumber of classes:", len(class_to_idx))

print("\nClass mapping:")
print(class_to_idx)

# --------------------------------------------------
# Step 2: Verify the expected split sizes
# --------------------------------------------------

expected_split_counts = {
    "train": 19440,
    "validation": 2160,
    "test": 5400
}

actual_split_counts = splits_df["split"].value_counts().to_dict()

assert len(splits_df) == 27000
assert actual_split_counts == expected_split_counts
assert len(class_to_idx) == 10

# --------------------------------------------------
# Step 3: Verify that all saved image paths exist
# --------------------------------------------------

image_paths_exist = splits_df["relative_path"].apply(
    lambda path: os.path.isfile(
        os.path.join(DATASET_DIR, path)
    )
)

assert image_paths_exist.all()

# --------------------------------------------------
# Step 4: Confirm successful loading and verification
# --------------------------------------------------

print("\nAll saved image paths exist:", image_paths_exist.all())

print("\nSaved split information loaded and verified successfully!")

#### 5.4.1 Image Preprocessing and Data Loading

We used PyTorch to load the EuroSAT images and their corresponding numerical labels. Each image was converted into a tensor and normalized using the ImageNet channel means and standard deviations, which are appropriate for our pretrained ResNet models.

We retained the original image resolution of 64 × 64 pixels. The images were then organized into batches using PyTorch DataLoaders. The training loader shuffles the training images, while the validation and test loaders preserve their order.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ImageNet normalization values for pretrained ResNet models
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Convert images to tensors and normalize them
image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# Define a custom EuroSAT dataset
class EuroSATDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = os.path.join(
            self.root_dir,
            row["relative_path"]
        )

        # Load the image and ensure it has three RGB channels
        image = Image.open(image_path).convert("RGB")

        # Apply preprocessing
        if self.transform is not None:
            image = self.transform(image)

        # Retrieve the numerical class label
        label = int(row["class_idx"])

        return image, label


# Separate the saved records into their existing splits
train_df = splits_df[splits_df["split"] == "train"]
val_df = splits_df[splits_df["split"] == "validation"]
test_df = splits_df[splits_df["split"] == "test"]

# Create dataset objects
train_dataset = EuroSATDataset(
    train_df, DATASET_DIR, image_transform
)
val_dataset = EuroSATDataset(
    val_df, DATASET_DIR, image_transform
)
test_dataset = EuroSATDataset(
    test_df, DATASET_DIR, image_transform
)

# Set the batch size
BATCH_SIZE = 64

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

# Verify the datasets and inspect one batch
print("DATASET SIZES")
print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

images, labels = next(iter(train_loader))

print("\nFIRST TRAINING BATCH")
print("Image tensor shape:", images.shape)
print("Label tensor shape:", labels.shape)
print("Image tensor data type:", images.dtype)
print("Label tensor data type:", labels.dtype)
print("Minimum normalized value:", images.min().item())
print("Maximum normalized value:", images.max().item())

print("\nPyTorch datasets and DataLoaders created successfully.")


# 6. ResNet-50 Baseline

## 6.1 Model Configuration and Training Strategy

We use an ImageNet-pretrained ResNet-50 as the baseline classifier for the ten EuroSAT land-use and land-cover classes. The final fully connected layer is replaced with a new layer containing ten output units, one for each class.

Training follows the two-stage transfer-learning strategy described in the reference paper. First, the pretrained feature extractor is frozen, and the newly initialized classification layer is trained using a learning rate of 0.01. Second, the entire network is fine-tuned using a lower learning rate in the range of 0.001–0.0001.

The AdamW optimizer is used for both stages as an experimental choice for this reimplementation. Other training settings, including the number of epochs and batch size, are documented separately where they are not explicitly specified in the original paper.

Model performance is monitored using the validation set, and the checkpoint with the highest validation accuracy is retained. Multiple random seeds are used to assess variability across training runs.

The test set remains separate from model training and selection and is reserved for final evaluation.

In [ ]:
import os
import random
import time
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path

# --------------------------------------------------
# 1. Reproducibility and training configuration
# --------------------------------------------------

# Use three random seeds to assess variability across runs
SEEDS = [42, 43, 44]

# Number of EuroSAT classes
NUM_CLASSES = 10

# Number of images processed in each training batch
BATCH_SIZE = 64

# Stage 1: Train only the new classification layer
# The paper specifies a learning rate of 0.01 for this stage.
# The number of epochs is an experimental choice.
HEAD_EPOCHS = 3
HEAD_LR = 0.01

# Stage 2: Fine-tune the entire ResNet-50 network
# The paper specifies a learning rate between 0.001 and 0.0001.
# We select 0.0001 for this experiment.
# The number of epochs is an experimental choice.
FINETUNE_EPOCHS = 7
FINETUNE_LR = 0.0001

# AdamW regularization
# The optimizer and weight decay are experimental choices.
WEIGHT_DECAY = 1e-4

# --------------------------------------------------
# 2. Select the computing device
# --------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# --------------------------------------------------
# 3. Load the class mapping from Section 5.3
# --------------------------------------------------

CLASS_MAPPING_PATH = Path("/content/class_to_idx.json")

if not CLASS_MAPPING_PATH.exists():
    raise FileNotFoundError(
        "class_to_idx.json was not found. "
        "Please run Section 5.3 first to create the class mapping."
    )

with open(CLASS_MAPPING_PATH, "r") as file:
    class_to_idx = json.load(file)

# Arrange class names according to their numerical indices
class_names = [
    name
    for name, index in sorted(
        class_to_idx.items(),
        key=lambda item: item[1]
    )
]

# --------------------------------------------------
# 4. Create the results directory
# --------------------------------------------------

RESULTS_DIR = Path("/content/eurosat_results/resnet50")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 5. Display the training configuration
# --------------------------------------------------

print("=" * 55)
print("RESNET-50 BASELINE CONFIGURATION")
print("=" * 55)

print("PyTorch version:", torch.__version__)
print("Torchvision version:", __import__("torchvision").__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU unavailable; training on CPU may be slow.")

print("\nNumber of classes:", len(class_names))
print("Class mapping:", class_to_idx)
print("Random seeds:", SEEDS)

print("\nStage 1: Classification head training")
print("Epochs:", HEAD_EPOCHS)
print("Learning rate:", HEAD_LR)

print("\nStage 2: Full-network fine-tuning")
print("Epochs:", FINETUNE_EPOCHS)
print("Learning rate:", FINETUNE_LR)

print("\nOptimizer: AdamW")
print("Weight decay:", WEIGHT_DECAY)
print("Batch size:", BATCH_SIZE)
print("Results directory:", RESULTS_DIR)

# --------------------------------------------------
# 6. Validate the prepared dataset splits
# --------------------------------------------------

# Confirm the split sizes match the 72/8/20 split
assert len(train_paths) == 19440, "Incorrect training split size."
assert len(val_paths) == 2160, "Incorrect validation split size."
assert len(test_paths) == 5400, "Incorrect test split size."

# Confirm the total number of images
assert (
    len(train_paths)
    + len(val_paths)
    + len(test_paths)
) == 27000, "Incorrect total number of images."

print("\nAll configuration and split-size checks passed!")


## 6.2 ResNet-50 Model and Training

In this section, we implement the ResNet-50 baseline for EuroSAT land-use and land-cover classification. We use an ImageNet-pretrained ResNet-50 model and replace its final fully connected layer to predict the ten EuroSAT classes.

Training is performed in two stages. First, we train the newly initialized classification head while keeping the pretrained feature-extraction layers frozen. Second, we fine-tune the entire network using a lower learning rate.

We use the AdamW optimizer with weight decay and evaluate model performance on the validation set after each epoch. The best model is selected based on validation accuracy. This process is repeated using three random seeds (42, 43, and 44) to assess the consistency of the results.

The test set is reserved for final evaluation after model selection. The experimental settings are documented as our implementation choices; they are not assumed to reproduce every detail of the original paper.


In [ ]:
import copy
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torchvision import models
from pathlib import Path


# ------------------------------------------------------------
# 1. Set the directory for saving results
# ------------------------------------------------------------

# Use the same results directory defined in the configuration cell.
# This keeps all ResNet-50 baseline outputs organized in one place.

RESULTS_DIR = Path("/content/eurosat_results/resnet50")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Set random seeds for reproducibility
# ------------------------------------------------------------

def set_seed(seed):
    """
    Set random seeds for Python, NumPy, and PyTorch.

    This helps make the training runs reproducible.
    """

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    # Set seed for CUDA when a GPU is available
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Configure cuDNN for deterministic operations where possible
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# 3. Build the ResNet-50 model
# ------------------------------------------------------------

def build_resnet50(num_classes):
    """
    Load an ImageNet-pretrained ResNet-50 model.

    Replace its final fully connected layer with a new layer
    that predicts the 10 EuroSAT land-use and land-cover classes.
    """

    # Load ResNet-50 with ImageNet-pretrained weights
    model = models.resnet50(
        weights=models.ResNet50_Weights.DEFAULT
    )

    # Get the number of input features to the original classifier
    num_features = model.fc.in_features

    # Replace the original classifier with a 10-class classifier
    model.fc = nn.Linear(
        num_features,
        num_classes
    )

    return model


# ------------------------------------------------------------
# 4. Define one training or evaluation epoch
# ------------------------------------------------------------

def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    train_head_only=False
):
    """
    Run one epoch of training or evaluation.

    Parameters
    ----------
    model:
        The ResNet-50 model.

    loader:
        Training or validation DataLoader.

    criterion:
        Loss function.

    optimizer:
        If provided, the model is trained.
        If None, the model is evaluated.

    train_head_only:
        If True, only the final classification layer is trained.
        The backbone remains in evaluation mode so that its
        BatchNorm running statistics are not updated.
    """

    # An optimizer indicates that this is a training epoch
    is_training = optimizer is not None

    # --------------------------------------------------------
    # Set model mode
    # --------------------------------------------------------

    if is_training:

        # Set the model to training mode
        model.train()

        # During Stage 1, freeze the backbone's BatchNorm layers.
        # Only the final classification layer is in training mode.
        if train_head_only:
            model.eval()
            model.fc.train()

    else:

        # During validation, all model layers are in evaluation mode
        model.eval()


    # --------------------------------------------------------
    # Initialize metrics
    # --------------------------------------------------------

    total_loss = 0.0
    total_correct = 0
    total_samples = 0


    # --------------------------------------------------------
    # Process batches
    # --------------------------------------------------------

    # Gradients are enabled during training and disabled during
    # validation to reduce memory usage and computation.
    with torch.set_grad_enabled(is_training):

        for images, labels in loader:

            # Move images and labels to the selected device
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            # Clear gradients before each training batch
            if is_training:
                optimizer.zero_grad()

            # Forward pass
            outputs = model(images)

            # Calculate classification loss
            loss = criterion(outputs, labels)

            # Backpropagation and parameter update
            if is_training:

                loss.backward()
                optimizer.step()


            # ------------------------------------------------
            # Calculate batch metrics
            # ------------------------------------------------

            # Predicted class is the class with the highest score
            predictions = outputs.argmax(dim=1)

            # Number of correct predictions in this batch
            correct = (predictions == labels).sum().item()

            # Number of samples in this batch
            batch_size = labels.size(0)

            # Accumulate loss and correct predictions
            total_loss += loss.item() * batch_size
            total_correct += correct
            total_samples += batch_size


    # --------------------------------------------------------
    # Calculate epoch metrics
    # --------------------------------------------------------

    # Average loss across all samples
    epoch_loss = total_loss / total_samples

    # Accuracy across all samples
    epoch_accuracy = total_correct / total_samples

    return epoch_loss, epoch_accuracy


# ------------------------------------------------------------
# 5. Train ResNet-50 using the two-stage training strategy
# ------------------------------------------------------------

def train_resnet50(seed):

    """
    Train one ResNet-50 model using a specified random seed.

    Stage 1:
        Freeze the backbone and train only the classifier.

    Stage 2:
        Fine-tune the entire network.

    The best model is selected based on validation accuracy.
    The test set is not used during training or model selection.
    """

    print("\n" + "=" * 70)
    print(f"Starting ResNet-50 training with seed: {seed}")
    print("=" * 70)


    # --------------------------------------------------------
    # 1. Set seed
    # --------------------------------------------------------

    set_seed(seed)


    # --------------------------------------------------------
    # 2. Initialize model and loss function
    # --------------------------------------------------------

    model = build_resnet50(NUM_CLASSES).to(DEVICE)

    # Cross-entropy loss for multi-class classification
    criterion = nn.CrossEntropyLoss()


    # --------------------------------------------------------
    # 3. Initialize model-selection variables
    # --------------------------------------------------------

    best_val_accuracy = -1.0
    best_state = None
    best_epoch = None

    # Store metrics from every epoch
    history = []

    # Global epoch counter across both training stages
    global_epoch = 0

    # Start measuring total training time
    start_time = time.time()


    # ========================================================
    # STAGE 1: Train only the classification head
    # ========================================================

    print("\nStage 1: Training the classification head")
    print(f"Epochs: {HEAD_EPOCHS}")
    print(f"Learning rate: {HEAD_LR}")


    # --------------------------------------------------------
    # Freeze all model parameters
    # --------------------------------------------------------

    for parameter in model.parameters():
        parameter.requires_grad = False


    # Unfreeze only the final fully connected layer
    for parameter in model.fc.parameters():
        parameter.requires_grad = True


    # --------------------------------------------------------
    # Define Stage 1 optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.fc.parameters(),
        lr=HEAD_LR,
        weight_decay=WEIGHT_DECAY
    )


    # --------------------------------------------------------
    # Stage 1 training loop
    # --------------------------------------------------------

    for epoch in range(HEAD_EPOCHS):

        global_epoch += 1

        # Train the classification head
        train_loss, train_accuracy = run_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            train_head_only=True
        )

        # Evaluate on the validation set
        val_loss, val_accuracy = run_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion
        )


        # Save metrics for this epoch
        history.append({
            "seed": seed,
            "epoch": global_epoch,
            "stage": "head_training",
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy
        })


        # Print epoch results
        print(
            f"Stage 1 | Epoch {epoch + 1}/{HEAD_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.4f}"
        )


        # ----------------------------------------------------
        # Save the best model based on validation accuracy
        # ----------------------------------------------------

        if val_accuracy > best_val_accuracy:

            best_val_accuracy = val_accuracy

            # Deep copy the model parameters
            best_state = copy.deepcopy(model.state_dict())

            # Record the epoch at which the best model was found
            best_epoch = global_epoch


    # ========================================================
    # STAGE 2: Fine-tune the entire network
    # ========================================================

    print("\nStage 2: Fine-tuning the entire network")
    print(f"Epochs: {FINETUNE_EPOCHS}")
    print(f"Learning rate: {FINETUNE_LR}")


    # --------------------------------------------------------
    # Unfreeze all model parameters
    # --------------------------------------------------------

    for parameter in model.parameters():
        parameter.requires_grad = True


    # --------------------------------------------------------
    # Define Stage 2 optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=FINETUNE_LR,
        weight_decay=WEIGHT_DECAY
    )


    # --------------------------------------------------------
    # Stage 2 training loop
    # --------------------------------------------------------

    for epoch in range(FINETUNE_EPOCHS):

        global_epoch += 1

        # Train the entire network
        train_loss, train_accuracy = run_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer
        )

        # Evaluate on the validation set
        val_loss, val_accuracy = run_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion
        )


        # Save metrics for this epoch
        history.append({
            "seed": seed,
            "epoch": global_epoch,
            "stage": "fine_tuning",
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy
        })


        # Print epoch results
        print(
            f"Stage 2 | Epoch {epoch + 1}/{FINETUNE_EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.4f}"
        )


        # ----------------------------------------------------
        # Save the best model based on validation accuracy
        # ----------------------------------------------------

        if val_accuracy > best_val_accuracy:

            best_val_accuracy = val_accuracy

            # Deep copy the model parameters
            best_state = copy.deepcopy(model.state_dict())

            # Record the epoch at which the best model was found
            best_epoch = global_epoch


    # ========================================================
    # 6. Restore the best model
    # ========================================================

    # The selected checkpoint is the one with the highest
    # validation accuracy across both training stages.
    model.load_state_dict(best_state)


    # ========================================================
    # 7. Save the trained model checkpoint
    # ========================================================

    checkpoint_path = RESULTS_DIR / f"resnet50_seed_{seed}.pth"

    torch.save(
        {
            "seed": seed,
            "state_dict": model.state_dict(),
            "best_val_accuracy": best_val_accuracy,
            "best_epoch": best_epoch,
            "class_names": list(class_to_idx.keys())
        },
        checkpoint_path
    )


    # ========================================================
    # 8. Calculate total training time
    # ========================================================

    elapsed_minutes = (time.time() - start_time) / 60


    print("\nTraining completed.")
    print(f"Best validation accuracy: {best_val_accuracy:.4f}")
    print(f"Best epoch: {best_epoch}")
    print(f"Training time: {elapsed_minutes:.2f} minutes")
    print(f"Checkpoint saved to: {checkpoint_path}")


    # ========================================================
    # 9. Return results
    # ========================================================

    return {
        "seed": seed,
        "best_val_accuracy": best_val_accuracy,
        "best_epoch": best_epoch,
        "training_minutes": elapsed_minutes,
        "checkpoint": str(checkpoint_path),
        "history": history
    }


# ============================================================
# 10. Run training for all configured random seeds
# ============================================================

all_run_results = []

for seed in SEEDS:

    run_result = train_resnet50(seed)

    all_run_results.append(run_result)


# ============================================================
# 11. Combine training histories
# ============================================================

# Collect history from all seeds into one DataFrame
all_history = []

for run_result in all_run_results:
    all_history.extend(run_result["history"])


history_df = pd.DataFrame(all_history)


# Save training history to CSV
history_path = RESULTS_DIR / "training_history.csv"

history_df.to_csv(
    history_path,
    index=False
)


# ============================================================
# 12. Summarize validation results across seeds
# ============================================================

validation_results_df = pd.DataFrame([
    {
        "seed": result["seed"],
        "best_val_accuracy": result["best_val_accuracy"],
        "best_epoch": result["best_epoch"],
        "training_minutes": result["training_minutes"],
        "checkpoint": result["checkpoint"]
    }
    for result in all_run_results
])


# Calculate mean and standard deviation of validation accuracy
mean_val_accuracy = validation_results_df[
    "best_val_accuracy"
].mean()

std_val_accuracy = validation_results_df[
    "best_val_accuracy"
].std(ddof=1)


# ============================================================
# 13. Display final results
# ============================================================

print("\n" + "=" * 70)
print("RESNET-50 BASELINE: VALIDATION RESULTS")
print("=" * 70)

display(validation_results_df)

print(
    f"\nMean validation accuracy: "
    f"{mean_val_accuracy:.4f} ± {std_val_accuracy:.4f}"
)

print(f"\nTraining history saved to: {history_path}")
print(f"Model checkpoints saved in: {RESULTS_DIR}")

### 6.3 Training Curves

Training and validation loss and accuracy are visualized for each of the three random seeds (42, 43, and 44). These curves help assess model convergence, generalization, and training stability across the two training stages.

The first stage trains only the classification head, while the second stage fine-tunes the full ResNet-50 network. Validation performance is used to select the best checkpoint for each seed. The test set is not used during training or checkpoint selection.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from pathlib import Path


# ------------------------------------------------------------
# 1. Load the training history
# ------------------------------------------------------------

# Use the same results directory as the ResNet-50 training cell.
# This directory contains the training history for seeds 42, 43, and 44.

RESULTS_DIR = Path("/content/eurosat_results/resnet50")

# Load the saved training history
history_path = RESULTS_DIR / "training_history.csv"

history_df = pd.read_csv(history_path)


# ------------------------------------------------------------
# 2. Display experiment information
# ------------------------------------------------------------

print("Experiment: ResNet-50")
print("Fine-tuning learning rate: 0.0001")
print("Seeds:", sorted(history_df["seed"].unique()))
print("Number of training records:", len(history_df))


# ------------------------------------------------------------
# 3. Check the required columns
# ------------------------------------------------------------

# These columns should have been saved by the training cell.
required_columns = {
    "seed",
    "epoch",
    "train_loss",
    "val_loss",
    "train_accuracy",
    "val_accuracy"
}

# Identify any missing columns
missing_columns = required_columns - set(history_df.columns)

# Stop execution if required columns are missing
assert not missing_columns, (
    f"Missing columns in training history: {missing_columns}"
)


# ------------------------------------------------------------
# 4. Prepare the data for plotting
# ------------------------------------------------------------

# Create a copy so the original DataFrame remains unchanged
plot_df = history_df.copy()

# Sort records by seed and epoch
plot_df = plot_df.sort_values(
    ["seed", "epoch"]
)

# Convert accuracy values to percentages
# This makes the accuracy plots easier to interpret.
plot_df["train_accuracy_pct"] = (
    plot_df["train_accuracy"] * 100
)

plot_df["val_accuracy_pct"] = (
    plot_df["val_accuracy"] * 100
)

# Get the random seeds included in the experiment
seeds = sorted(
    plot_df["seed"].unique()
)


# ============================================================
# 5. Plot training and validation loss
# ============================================================

plt.figure(figsize=(11, 6))


# Plot the loss curves for each seed
for seed in seeds:

    # Select the history for the current seed
    seed_df = plot_df[
        plot_df["seed"] == seed
    ]

    # Training loss
    plt.plot(
        seed_df["epoch"],
        seed_df["train_loss"],
        marker="o",
        label=f"Seed {seed} — Training"
    )

    # Validation loss
    plt.plot(
        seed_df["epoch"],
        seed_df["val_loss"],
        marker="s",
        linestyle="--",
        label=f"Seed {seed} — Validation"
    )


# Mark the transition between the two training stages
# Stage 1: Classification head training
# Stage 2: Full-network fine-tuning

plt.axvline(
    x=HEAD_EPOCHS + 0.5,
    linestyle=":",
    label="Stage transition"
)


# Set plot labels and title
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "ResNet-50: Training and Validation Loss"
)


# Display epochs from 1 to the total number of epochs
plt.xticks(
    range(
        1,
        HEAD_EPOCHS + FINETUNE_EPOCHS + 1
    )
)


# Display legend
plt.legend(
    ncol=2,
    fontsize=9
)


# Add grid and adjust layout
plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

# Display the plot
plt.show()


# ============================================================
# 6. Plot training and validation accuracy
# ============================================================

plt.figure(figsize=(11, 6))


# Plot the accuracy curves for each seed
for seed in seeds:

    # Select the history for the current seed
    seed_df = plot_df[
        plot_df["seed"] == seed
    ]

    # Training accuracy
    plt.plot(
        seed_df["epoch"],
        seed_df["train_accuracy_pct"],
        marker="o",
        label=f"Seed {seed} — Training"
    )

    # Validation accuracy
    plt.plot(
        seed_df["epoch"],
        seed_df["val_accuracy_pct"],
        marker="s",
        linestyle="--",
        label=f"Seed {seed} — Validation"
    )


# Mark the transition between the two training stages
plt.axvline(
    x=HEAD_EPOCHS + 0.5,
    linestyle=":",
    label="Stage transition"
)


# Set plot labels and title
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")

plt.title(
    "ResNet-50: Training and Validation Accuracy"
)


# Display epochs from 1 to the total number of epochs
plt.xticks(
    range(
        1,
        HEAD_EPOCHS + FINETUNE_EPOCHS + 1
    )
)


# Display legend
plt.legend(
    ncol=2,
    fontsize=9
)


# Add grid and adjust layout
plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

# Display the plot
plt.show()


# ============================================================
# 7. Confirm completion
# ============================================================

print("\nTraining-curve visualizations completed.")

print("Seeds plotted:", seeds)

print("Stage 1: Classification head training")
print("Stage 2: Full-network fine-tuning")

print(f"Training history loaded from: {history_path}")

### 6.4 Test Set Evaluation

The best ResNet-50 checkpoint for each random seed was selected using validation accuracy. The selected checkpoints are evaluated on the held-out test set to assess their generalization performance.

The test set is not used for training, hyperparameter selection, or checkpoint selection. Accuracy, precision, recall, and F1-score are reported for each seed, along with the mean and standard deviation across the three runs.

In [ ]:
# ============================================================
# 6.4 Test Set Evaluation
# ============================================================

import torch
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from torchvision import models
import torch.nn as nn
from pathlib import Path


# ------------------------------------------------------------
# 1. Set results directory
# ------------------------------------------------------------

RESULTS_DIR = Path("/content/eurosat_results/resnet50")


# ------------------------------------------------------------
# 2. Evaluate one saved model on the test set
# ------------------------------------------------------------

def evaluate_resnet50(checkpoint_path):

    # Load the saved checkpoint
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE
    )

    # Rebuild the ResNet-50 architecture
    model = models.resnet50(
        weights=None
    )

    # Replace final classification layer
    model.fc = nn.Linear(
        model.fc.in_features,
        NUM_CLASSES
    )

    # Load trained model parameters
    model.load_state_dict(
        checkpoint["state_dict"]
    )

    # Move model to GPU or CPU
    model = model.to(DEVICE)

    # Set model to evaluation mode
    model.eval()

    all_predictions = []
    all_labels = []

    # Disable gradient computation during evaluation
    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            # Forward pass
            outputs = model(images)

            # Get predicted class
            predictions = outputs.argmax(dim=1)

            # Store predictions and true labels
            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    # Calculate evaluation metrics
    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision = precision_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "predictions": all_predictions,
        "labels": all_labels
    }


# ------------------------------------------------------------
# 3. Evaluate all three seeds
# ------------------------------------------------------------

test_results = []
test_predictions = {}

for seed in SEEDS:

    checkpoint_path = (
        RESULTS_DIR / f"resnet50_seed_{seed}.pth"
    )

    print(f"\nEvaluating seed {seed}")

    result = evaluate_resnet50(
        checkpoint_path
    )

    # Store metrics
    test_results.append({
        "seed": seed,
        "test_accuracy": result["accuracy"],
        "test_precision": result["precision"],
        "test_recall": result["recall"],
        "test_f1_score": result["f1_score"]
    })

    # Store predictions and labels for later analysis
    test_predictions[seed] = result

    print(
        f"Test Accuracy: {result['accuracy']:.4f}"
    )

    print(
        f"Test Precision: {result['precision']:.4f}"
    )

    print(
        f"Test Recall: {result['recall']:.4f}"
    )

    print(
        f"Test F1-score: {result['f1_score']:.4f}"
    )


# ------------------------------------------------------------
# 4. Create results table
# ------------------------------------------------------------

test_results_df = pd.DataFrame(
    test_results
)

display(test_results_df)


# ------------------------------------------------------------
# 5. Calculate mean and standard deviation
# ------------------------------------------------------------

metrics = [
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1_score"
]

summary_df = test_results_df[
    metrics
].agg(["mean", "std"])

print("\nMean and standard deviation across seeds:")
display(summary_df)


# ------------------------------------------------------------
# 6. Save evaluation results
# ------------------------------------------------------------

test_results_path = (
    RESULTS_DIR / "test_results.csv"
)

test_results_df.to_csv(
    test_results_path,
    index=False
)

print(
    f"\nTest results saved to: {test_results_path}"
)

### 6.5 Baseline Results Summary

The ResNet-50 baseline was evaluated on the held-out test set using three random seeds (42, 43, and 44). The model achieved a mean test accuracy of 96.32% ± 0.04%, a mean precision of 96.19% ± 0.03%, a mean recall of 96.20% ± 0.05%, and a mean F1-score of 96.18% ± 0.04%.

The low standard deviations across the three runs indicate that the baseline performance was consistent across random seeds. The results provide a reference point for the subsequent experiments, including the label-efficiency study (Part A) and the smaller ResNet-18 model (Part B).

The reproduced accuracy is lower than the 98.57% reported in the original EuroSAT paper. This difference should be interpreted in the context of the experimental setup, including the validation split and training choices made in this reproduction. The baseline results will be retained for comparison with the later experiments.

In [ ]:
import pandas as pd

# Baseline test results across three seeds
baseline_results = pd.DataFrame({
    "Seed": [42, 43, 44],
    "Test Accuracy (%)": [96.2778, 96.3519, 96.3333],
    "Test Precision (%)": [96.1601, 96.2074, 96.2083],
    "Test Recall (%)": [96.1483, 96.2100, 96.2367],
    "Test F1-score (%)": [96.1407, 96.1982, 96.2120]
})

# Add mean and standard deviation rows
mean_row = {
    "Seed": "Mean",
    "Test Accuracy (%)": 96.3210,
    "Test Precision (%)": 96.1920,
    "Test Recall (%)": 96.1983,
    "Test F1-score (%)": 96.1836
}

std_row = {
    "Seed": "Std",
    "Test Accuracy (%)": 0.0385,
    "Test Precision (%)": 0.0276,
    "Test Recall (%)": 0.0453,
    "Test F1-score (%)": 0.0378
}

baseline_results = pd.concat(
    [baseline_results, pd.DataFrame([mean_row, std_row])],
    ignore_index=True
)

# Display the table
display(baseline_results.round(4))